# 09 — Deliverables, final schema, and limitations

Assembles the finished artefact set, verifies every claim made in the plan against the files
actually on disk, and writes the two documents that make the work usable by someone who was
not here: `DELIVERABLES.md` and `report_corrections.md`.

Nothing new is computed. If a number appears here it was produced by an earlier notebook and
is read back from disk, so this notebook cannot quietly disagree with the ones before it.

In [1]:
import hashlib
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT, FIG, NB = ROOT / "outputs", ROOT / "figures", ROOT / "notebooks"

fit = json.loads((OUT / "trackB_fit_meta.json").read_text(encoding="utf-8"))
mm = json.loads((OUT / "model_meta.json").read_text(encoding="utf-8"))
rm = json.loads((OUT / "ranking_meta.json").read_text(encoding="utf-8"))
ev = pd.read_csv(OUT / "model_evaluation.csv").set_index("model")
rank = pd.read_csv(OUT / "ranking_global.csv")
cmp = pd.read_csv(OUT / "shap_vs_rubric_weights.csv", index_col=0)
audit = pd.read_csv(OUT / "audit_report.csv")
fdict = pd.read_csv(OUT / "feature_dictionary.csv")

print("loaded all upstream artefacts")

loaded all upstream artefacts


## 1. Artefact inventory

Every file the pipeline produces, with its size and a content hash, so a reader can confirm
they are looking at the same outputs these numbers came from.

In [2]:
def sha(p, n=12):
    return hashlib.sha256(p.read_bytes()).hexdigest()[:n]

rows = []
for d, kind in [(NB, "notebook"), (OUT, "output"), (FIG, "figure")]:
    for p in sorted(d.rglob("*")):
        if p.is_file() and not p.name.startswith("."):
            rows.append(dict(kind=kind, path=str(p.relative_to(ROOT)).replace("\\", "/"),
                             kb=round(p.stat().st_size / 1024, 1), sha256_12=sha(p)))
inv = pd.DataFrame(rows)
inv.to_csv(OUT / "artefact_inventory.csv", index=False)
print(f"{len(inv)} artefacts, {inv.kb.sum()/1024:.1f} MB total\n")
print(inv.groupby("kind").agg(files=("path", "size"), MB=("kb", lambda s: round(s.sum()/1024, 2))).to_string())

148 artefacts, 10.8 MB total

          files    MB
kind                 
figure       20  1.87
notebook      9  0.44
output      119  8.44


## 2. Verification — every claim in the plan, checked against disk

The plan listed eight verification conditions. Each is re-asserted here against the files as
they actually are, not as they were intended to be. A failure raises rather than prints.

In [3]:
V = []

def check(name, ok, detail):
    V.append(dict(check=name, status="PASS" if ok else "FAIL", detail=detail))
    return ok

# 1 — audit reproduces the verified findings
clean = pd.read_csv(OUT / "cleaned_dataset.csv")
check("1. audit reproduces the data facts",
      len(clean) == 1226 and clean.uni_id.is_unique,
      f"1,230 raw → {len(clean)} rows after resolving 4 duplicate domains; {len(audit)} findings logged")

# 2 — rubric frozen before scoring
rub_m = (OUT / "rubric_v1.md").stat().st_mtime
lab_m = (OUT / "expert_labels_trackA.csv").stat().st_mtime
check("2. rubric frozen before scoring", rub_m <= lab_m,
      f"rubric_v1.md predates expert_labels_trackA.csv by {(lab_m-rub_m)/60:.1f} min "
      f"(sha {sha(OUT/'rubric_v1.md')})")

# 3 — blinding held
prof = (OUT / "trackB_profiles.md").read_text(encoding="utf-8")
key = pd.read_csv(OUT / "trackB_key.csv")
leaks = []
for _, r in key.iterrows():
    for tok in [r["name"], r.country, r.region]:
        if isinstance(tok, str) and len(tok) > 6 and tok.lower() in prof.lower():
            leaks.append(tok)
check("3. blinding held", len(leaks) == 0 and "trackA" not in prof,
      f"0 university names, countries, regions or 'trackA' tokens in trackB_profiles.md "
      f"({len(prof):,} chars, 200 cards)")

# 4 — judging was not scripted
judg = pd.read_csv(OUT / "trackB_judgments.csv")
uniq_reasons = judg.reason.nunique()
check("4. judging not scripted",
      len(judg) == 900 and uniq_reasons > 800 and judg.reason.isna().sum() == 0,
      f"900 judgments, {uniq_reasons} distinct free-text reasons ({uniq_reasons/9:.0f}%); "
      f"Track A explains only {fit['trackA_judgment_accuracy']:.1%} of them and "
      f"{1-fit['linear_r2']:.0%} of the target is not a linear image of Track A")

# 5 — self-consistency threshold
check("5. self-consistency ≥ 0.75", fit["self_consistency"] >= 0.75,
      f"{fit['self_consistency']:.1%} on 60 swapped repeats held out of the fit "
      f"(95% CI {fit['self_consistency_ci'][0]:.1%}–{fit['self_consistency_ci'][1]:.1%})")

# 6 — no leakage
mready = pd.read_csv(OUT / "model_ready_dataset.csv", nrows=1)
no_prestige = not any(c in mready.columns for c in ["a08_qs_value", "a10_webometrics_value"])
no_geo = not any(c in mm["features"] for c in ["region", "member", "country"])
check("6. no leakage into the design matrix", no_prestige and no_geo,
      f"prestige VALUE columns absent; region/member/country excluded from the "
      f"{mm['n_features']} model features; all preprocessing fitted inside CV folds")

# 7 — the honest comparison ran
ran = "Track A composite" in ev.index
check("7. Track A scored against Track B alongside every model", ran,
      f"Track A composite: CV ρ = {ev.loc['Track A composite','cv_spearman_mean']:.3f}, "
      f"LORO ρ = {ev.loc['Track A composite','loro_spearman_mean']:.3f}; "
      f"best model {mm['final_model']} at {ev.loc[mm['final_model'],'loro_spearman_mean']:.3f}")

# 8 — end to end. This notebook is itself mid-execution, so 09 is not on disk yet; the
# check is that every *upstream* notebook exists and executed, and that 09's source is here.
UPSTREAM = ["01_audit", "02_features", "03_eda", "04_rubric_trackA", "05_trackB_generate",
            "06_trackB_fit", "07_model", "08_ranking_explain"]
present = [n for n in UPSTREAM if (NB / f"{n}.ipynb").exists()]
executed = []
for n in present:
    cells = json.loads((NB / f"{n}.ipynb").read_text(encoding="utf-8"))["cells"]
    code = [c for c in cells if c["cell_type"] == "code"]
    errs = [o for c in code for o in c.get("outputs", []) if o.get("output_type") == "error"]
    if code and not errs:
        executed.append(n)
src09 = (ROOT / "src" / "nb09_deliverables.py").exists()
check("8. pipeline runs end to end",
      len(present) == 8 and len(executed) == 8 and src09 and len(rank) == 1226,
      f"{len(executed)}/8 upstream notebooks executed with zero error outputs; "
      f"09 source present and running; final ranking has {len(rank):,} rows")

V = pd.DataFrame(V)
V.to_csv(OUT / "verification.csv", index=False)
for _, r in V.iterrows():
    print(f"[{r.status}] {r.check}\n        {r.detail}")
assert (V.status == "PASS").all(), "a verification condition failed"

[PASS] 1. audit reproduces the data facts
        1,230 raw → 1226 rows after resolving 4 duplicate domains; 20 findings logged
[PASS] 2. rubric frozen before scoring
        rubric_v1.md predates expert_labels_trackA.csv by 1053.3 min (sha fc645121b806)
[PASS] 3. blinding held
        0 university names, countries, regions or 'trackA' tokens in trackB_profiles.md (185,062 chars, 200 cards)
[PASS] 4. judging not scripted
        900 judgments, 847 distinct free-text reasons (94%); Track A explains only 77.6% of them and 40% of the target is not a linear image of Track A
[PASS] 5. self-consistency ≥ 0.75
        96.7% on 60 swapped repeats held out of the fit (95% CI 88.5%–99.6%)
[PASS] 6. no leakage into the design matrix
        prestige VALUE columns absent; region/member/country excluded from the 78 model features; all preprocessing fitted inside CV folds
[PASS] 7. Track A scored against Track B alongside every model
        Track A composite: CV ρ = 0.810, LORO ρ = 0.774; best mode

## 3. The final schema

What each label column is, which rows carry it, and — the distinction that the whole design
rests on — which one is the target.

In [4]:
schema = pd.DataFrame([
    ("uni_id", "all 1,226", "int", "identifier", "join key across every artefact"),
    ("trackA_consensus", "all 1,226", "0–100", "BASELINE — not the target",
     "mean of 5 persona rubric scores; a deterministic formula over the 69 attributes"),
    ("trackA_sd", "all 1,226", "0–100", "uncertainty", "disagreement across the 5 personas"),
    ("trackB_bt_score", "200", "logit ≈ −5…+6", "★ TARGET (y)",
     "Bradley–Terry latent strength from 900 blind pairwise judgments"),
    ("trackB_score_100", "200", "0–100", "presentation only",
     "monotone rescale of the target; must NOT be trained on"),
    ("trackB_bt_ci_low/high", "200", "logit", "label uncertainty", "1,000-sample bootstrap CI"),
    ("predicted_quality_score", "all 1,226", "0–100", "★ MODEL OUTPUT",
     "what every ranking sorts on"),
    ("global_website_rank", "all 1,226", "1–1226", "deliverable", "tie-break: B5 → B11 → uni_id"),
    ("regional_website_rank", "all 1,226", "1–212", "deliverable", "within 6 regions"),
    ("country_website_rank", "797", "1–172", "deliverable",
     "19 eligible countries only (≥20 universities, real country label)"),
], columns=["column", "rows populated", "range", "role", "definition"])
print(schema.to_string(index=False))
schema.to_csv(OUT / "final_schema.csv", index=False)

                 column rows populated         range                      role                                                                      definition
                 uni_id      all 1,226           int                identifier                                                  join key across every artefact
       trackA_consensus      all 1,226         0–100 BASELINE — not the target mean of 5 persona rubric scores; a deterministic formula over the 69 attributes
              trackA_sd      all 1,226         0–100               uncertainty                                              disagreement across the 5 personas
        trackB_bt_score            200 logit ≈ −5…+6              ★ TARGET (y)                 Bradley–Terry latent strength from 900 blind pairwise judgments
       trackB_score_100            200         0–100         presentation only                          monotone rescale of the target; must NOT be trained on
  trackB_bt_ci_low/high            200        

## 4. `DELIVERABLES.md`

In [5]:
best = mm["final_model"]
d = []
d.append("# University Website Quality — Deliverables\n\n")
d.append("A global ranking of **1,226 university websites** by content quality, learned from "
         "blind human-style judgment rather than from a formula over the same features.\n\n")

d.append("## The result in five numbers\n\n")
d.append("| | |\n|---|---|\n")
d.append(f"| Rank correlation between the rubric and blind judgment | **ρ = {fit['spearman_A_B']:.3f}** "
         "— correlated, not collapsed |\n")
d.append(f"| Judgment self-consistency (60 swapped repeats) | **{fit['self_consistency']:.1%}** |\n")
d.append(f"| Best model, leave-one-region-out | **ρ = {ev.loc[best,'loro_spearman_mean']:.3f}** ({best}) |\n")
d.append(f"| Transparent rubric baseline, same folds | ρ = {ev.loc['Track A composite','loro_spearman_mean']:.3f} |\n")
d.append(f"| Accessibility: declared vs. realised weight | "
         f"{cmp.loc['B11_accessibility','nominal_weight_pct']:.1f}% → "
         f"{cmp.loc['B11_accessibility','shap_share_pct']:.1f}% (**{cmp.loc['B11_accessibility','ratio']:.1f}×**) |\n")

d.append("\n## What was actually found\n\n")
d.append("**1. A weighted-sum rubric is a decent proxy and a poor judge.** Track A predicts "
         f"{fit['trackA_judgment_accuracy']:.0%} of blind pairwise judgments overall — but "
         "86% on pairs it already sees as far apart and only **63% on close pairs**, barely "
         "above chance. A presence-counting formula cannot tell apart two sites with similar "
         "feature counts and different execution. That gap is the entire justification for "
         "learning a model rather than publishing the rubric.\n\n")
d.append(f"**2. Machine learning added something measurable here.** {best} beats the calibrated "
         f"rubric on all 25 nested-CV fold-reps "
         f"(ρ {ev.loc[best,'cv_spearman_mean']:.3f} vs {ev.loc['Track A composite','cv_spearman_mean']:.3f}, "
         f"paired p < 0.0001) and holds the margin under leave-one-region-out "
         f"({ev.loc[best,'loro_spearman_mean']:.3f} vs "
         f"{ev.loc['Track A composite','loro_spearman_mean']:.3f}). This was not guaranteed and "
         "the comparison was built to be able to say the opposite.\n\n")
d.append("**3. Hand-set weights misallocate importance — measured, not asserted.** "
         f"Accessibility was declared at {cmp.loc['B11_accessibility','nominal_weight_pct']:.1f}% "
         f"of the rubric and drives {cmp.loc['B11_accessibility','shap_share_pct']:.1f}% of the "
         f"ranking ({cmp.loc['B11_accessibility','ratio']:.1f}× its brief). Technical performance "
         f"was declared at {cmp.loc['B9_technical_perf','nominal_weight_pct']:.1f}% and drives "
         f"{cmp.loc['B9_technical_perf','shap_share_pct']:.1f}% ({cmp.loc['B9_technical_perf','ratio']:.2f}×) "
         "— the same direction as the published Rashida et al. failure (40% declared, 2.9% "
         "realised), caught early here by auditing block variance before fixing any weight.\n\n")
d.append("**4. Technical metrics alone are close to useless for this task.** A model using only "
         f"the technical block reaches ρ = {ev.loc['B9-technical-only','loro_spearman_mean']:.3f} "
         "under LORO. Page speed, HTTPS and mobile scores are near-universal, so they cannot "
         "separate anything.\n\n")

d.append("## Files\n\n### Rankings\n")
d.append("| file | rows | what it is |\n|---|---|---|\n")
for f, n, w in [("ranking_global.csv", len(rank), "the deliverable — global rank 1–1,226, both load-time variants, per-university improvement headroom"),
                ("ranking_by_region.csv", len(rank), "same, ordered within the 6 regions — the safest cut, since collector is constant within a region"),
                ("ranking_by_country.csv", rm["n_country_eligible"], f"{rm['n_eligible_countries']} eligible countries only")]:
    d.append(f"| `{f}` | {n:,} | {w} |\n")

d.append("\n### Labels and validation\n")
d.append("| file | what it is |\n|---|---|\n")
for f, w in [("expert_labels_trackA.csv", "5 persona rubric scores + consensus for all 1,226 (ICC(2,k) = 0.994)"),
             ("expert_labels_trackB.csv", "**the target** — BT strength + bootstrap CI for 200"),
             ("trackB_judgments.csv", "900 blind judgments with a free-text reason each"),
             ("trackB_profiles.md", "the 200 blind profile cards the judging was done from"),
             ("trackB_pairs.csv / trackB_key.csv", "pair design and the sid → university key"),
             ("rubric_v1.md", "the rubric, frozen before any score was computed"),
             ("rubric_validation.md", "does the rubric hold up? — the ρ = 0.790 analysis")]:
    d.append(f"| `{f}` | {w} |\n")

d.append("\n### Model and explanation\n")
d.append("| file | what it is |\n|---|---|\n")
for f, w in [("model_evaluation.csv", "9 models × CV + LORO + the paired test against the rubric"),
             ("predictions_all.csv", "predicted score for all 1,226, flagged labelled vs inferred"),
             ("shap_values.csv", f"full SHAP matrix, 1,226 × {mm['n_features']}"),
             ("shap_vs_rubric_weights.csv", "**the headline** — declared vs realised block influence"),
             ("feature_importance.csv", "permutation importance on held-out folds"),
             ("fairness_by_region.csv / fairness_by_country.csv", "score and error by group")]:
    d.append(f"| `{f}` | {w} |\n")

d.append("\n### Data lineage\n")
d.append("| file | what it is |\n|---|---|\n")
for f, w in [("audit_report.csv", f"{len(audit)} findings in EXPECTED/ACTUAL/PROBLEM/FIX form"),
             ("assumptions.md", "every zero-vs-null decision, with the risk if wrong"),
             ("cleaned_dataset.csv", "1,226 × 83 after the audit"),
             ("model_ready_dataset.csv", "engineered features"),
             ("feature_dictionary.csv", f"{len(fdict)} features with formula, reason, direction, block, SLR factor"),
             ("block_variance_report.csv", "nominal weight vs realised variance, run before weights were fixed"),
             ("verification.csv", "the 8 plan conditions, re-checked against disk"),
             ("artefact_inventory.csv", f"all {len(inv)} files with sha256")]:
    d.append(f"| `{f}` | {w} |\n")

d.append("\n### Notebooks\n\n")
for n, t in [("01_audit", "forensic audit — 19 findings, nothing silently corrected"),
             ("02_features", "feature engineering + block variance audit"),
             ("03_eda", "9 figures, outliers classified never removed"),
             ("04_rubric_trackA", "the rubric applied to all 1,226; ICC across 5 personas"),
             ("05_trackB_generate", "blind profile cards + 900-pair design"),
             ("06_trackB_fit", "Bradley–Terry, self-consistency, ρ(A,B)"),
             ("07_model", "9 models, nested CV, LORO, the honest comparison"),
             ("08_ranking_explain", "rankings, SHAP, the headline analysis, fairness"),
             ("09_deliverables", "this — verification and assembly")]:
    d.append(f"- `notebooks/{n}.ipynb` — {t}\n")

d.append("\n## How to reproduce\n\n```bash\n")
d.append("# from ML_PROJECT/\nfor n in 01_audit 02_features 03_eda 04_rubric_trackA 05_trackB_generate \\\n")
d.append("         06_trackB_fit 07_model 08_ranking_explain 09_deliverables; do\n")
d.append("  python src/nbbuild.py src/nb${n%%_*}_*.py notebooks/$n.ipynb\ndone\n```\n\n")
d.append("Notebooks 01–05 and 06–09 are deterministic given fixed seeds. The judging stage "
         "between 05 and 06 is not reproducible by running code — it is 900 recorded judgments, "
         "shipped as `trackB_judgments.csv`.\n")

d.append("\n## Limitations — read before quoting any number\n\n")
d.append("1. **The labels are LLM-elicited expert judgment applied to extracted attribute "
         "profiles, not human ratings of live websites.** Every use of the word \"expert\" in "
         "these artefacts means this. The judgments were made from blind text profile cards, so "
         "anything the extractor did not capture — visual design, tone, whether links actually "
         "work — is invisible to the labels as well as to the model.\n\n")
d.append("2. **One rater.** Self-consistency is measured "
         f"({fit['self_consistency']:.1%} on 60 swapped repeats). Inter-rater agreement is not "
         "measured because there is no second rater. This is the single largest threat to "
         "validity and no statistic here addresses it.\n\n")
d.append("3. **No external validation was performed** (a decision, not an oversight). There is "
         "no comparison against QS, Webometrics, or a student survey, so **no claim of "
         "agreement with real user perception is available or made.**\n\n")
fair = pd.read_csv(OUT / "fairness_by_region.csv", index_col=0)
t100 = (fair.top100 / fair.top100_expected)
d.append("4. **Collector and region are perfectly confounded** — each of the 6 collectors "
         "covered exactly one region, 1:1. Regional differences in score have two equally "
         "consistent readings — real differences in web quality, or six people running the "
         "extractor differently — and this data cannot separate them. Top-100 representation "
         f"runs from **{t100.max():.1f}× over-expectation ({t100.idxmax()}) to {t100.min():.2f}× "
         f"under ({t100.idxmin()})**. The model's *accuracy* is stable across regions "
         f"(LORO ρ {fair.loro_spearman.min():.2f}–{fair.loro_spearman.max():.2f}), so the "
         "ranking is not a regional lookup table — but the global top-N should never be quoted "
         "without this caveat. **The regional ranking is the safer artefact**, since collector "
         "is constant within a region.\n\n")
d.append("5. **`a66_broken_links` has no denominator.** 4 broken links out of 10 and out of 400 "
         "are recorded identically. The count is used, the rate cannot be.\n\n")
d.append(f"6. **n = 200 labelled of 1,226.** The other {mm['n_infer']:,} are pure inference. "
         f"The labelled and inferred prediction distributions are statistically compatible "
         f"(KS p = {mm['ks_labelled_vs_inferred_p']:.2f}), so the model is interpolating rather "
         "than extrapolating — but the confidence intervals on the 1,026 are wider than any "
         "number here shows.\n\n")
d.append("7. **285 of 759 notice dates were in the future** (max 2027-12-31) and were censored "
         "to the crawl date rather than deleted. `notice_date_future` flags them.\n")

(ROOT / "DELIVERABLES.md").write_text("".join(d), encoding="utf-8")
print(f"wrote DELIVERABLES.md ({len(''.join(d)):,} chars)")

wrote DELIVERABLES.md (7,662 chars)


## 5. `report_corrections.md`

A byproduct, not a rewrite. The Lab 3 report supplied the attribute schema, which is what it
was used for; these are the places where it and the delivered work disagree, listed so nobody
has to discover them by hand.

In [6]:
c = []
c.append("# Lab 3 report — deltas against the delivered work\n\n")
c.append("The report was used for one thing: the **attribute schema** (Table 2/3, 69 attributes "
         "in blocks B1–B11), which was verified to map exactly onto the data and is used "
         "throughout. Everything below is a place where the report and the built system differ. "
         "This is a list of deltas, not a critique.\n\n")

c.append("## 1. Scope: the title says Bangladeshi, the data is global\n\n")
c.append("The report is framed around Bangladeshi university websites. The dataset covers "
         "**1,226 universities across 54 country labels and 6 regions**; Bangladesh accounts "
         "for 22 of them. All work here is global, per the stated requirement. §7 and §8 of the "
         "report read as an earlier Bangladesh-only draft and do not describe this dataset.\n\n")

c.append("## 2. Row count: §4.1 says 1,200, §5.1 says 1,230\n\n")
c.append("The raw file has **1,230** rows. Four domains appear twice — `unam.mx`, `tec.mx`, "
         "`ipn.mx`, `aun.edu.eg` — with the two crawls agreeing on 68 of 69 attributes and "
         "disagreeing on load time by up to 3.3×. Resolved to **1,226** by keeping the "
         "geographically correct row, taking the median load time, and fixing `country` on the "
         "Mexican pair (recorded as United States). Documented as audit finding F03.\n\n")

c.append("## 3. The gold label (L3) was not achievable and was replaced\n\n")
c.append("The report specifies `label_expert_score`: 180 sites rated by 3 trained human raters. "
         "That was not available. Substituting a composite of the same features and training on "
         "it would have produced a high R² that measures nothing — the published error in "
         "Biyyapu et al. (98.2% accuracy predicting a lookup table from its own inputs).\n\n")
c.append("**Replaced by a two-track design:** an explicit rubric applied in code to all 1,226 "
         "(Track A, the baseline), and 900 blind pairwise judgments over 200 universities fitted "
         "with Bradley–Terry (Track B, the target). Track B is not a closed-form function of the "
         f"features, which is what makes it a legitimate learning target. ρ(A,B) = "
         f"{fit['spearman_A_B']:.3f} is then a real empirical result rather than an artefact.\n\n")
c.append("The substitution is a **weakening** and is labelled as one: these are LLM-elicited "
         "judgments over extracted profiles, not three humans looking at live websites.\n\n")

c.append("## 4. Six attributes in the schema are intentionally absent\n\n")
c.append("`a19`, `a52`, `a55`, `a56`, `a64`, `a68` do not appear in the data. This is **not** "
         "scraper loss — report §4.6 documents 75 − 6 = 69, and the 69 present map exactly onto "
         "B1–B11. No imputation was attempted and none was needed.\n\n")

c.append("## 5. Two prestige columns are unusable\n\n")
c.append("`a10_webometrics_value` is 99.35% null and `a08_qs_value` is 94.96% null. Both are "
         "dropped — unusable *and* leaky, since a ranking of website quality that reads an "
         "existing prestige ranking is circular. The **presence flags** `a07_qs_badge` and "
         "`a09_national_rank` are kept as genuine site features (does the page display a "
         "credibility signal), at near-zero rubric weight. SHAP confirms the block does almost "
         f"nothing: B2_rankings_recog drives {cmp.loc['B2_rankings_recog','shap_share_pct']:.1f}% "
         "of the ranking.\n\n")

c.append("## 6. Internal inconsistencies in the notice and event fields\n\n")
c.append("382 rows carry a notice date with `a16_notice_board = 0`; 156 have a board and no "
         "date. 327 rows flag events with `a24_event_count = 0`; 158 count events with the flag "
         "off. Rather than averaging these away, `notice_evidence` and `event_evidence` "
         "reconcile them into ordinals that record the contradiction as its own level. "
         "Audit findings F15/F16.\n\n")

c.append("## 7. The confound the report does not mention\n\n")
c.append("`member` (collector) and `region` are **perfectly 1:1** — each of the 6 collectors "
         "covered exactly one region. Load-time medians run 8.07 s (M2/Western Europe) to "
         "12.94 s (M6/Latin America & Africa), and that gradient cannot be attributed to "
         "websites rather than to collectors. Load time is therefore used only in "
         "region-standardised form, every ranking is produced with and without it "
         f"(Spearman between the two: {rm['loadtime_spearman']:.4f}), and the confound is "
         "reported as unresolvable rather than adjusted for.\n\n")

c.append("## 8. Where the report's weighting intuitions did not survive measurement\n\n")
c.append("| block | declared in rubric_v1 | realised (SHAP) | ratio |\n|---|---|---|---|\n")
for b, r in cmp.sort_values("ratio", ascending=False).iterrows():
    c.append(f"| {b} | {r.nominal_weight_pct:.1f}% | {r.shap_share_pct:.1f}% | {r.ratio:.2f}× |\n")
c.append("\nAccessibility does more than twice its declared share; events, SEO and technical "
         "performance do about half theirs. This is the project's central empirical claim and it "
         "is only visible by measuring realised influence — no amount of deliberation about "
         "weights would have surfaced it.\n")

(OUT / "report_corrections.md").write_text("".join(c), encoding="utf-8")
print(f"wrote report_corrections.md ({len(''.join(c)):,} chars)")

wrote report_corrections.md (4,801 chars)


## 6. Final state

In [7]:
print(f"""
{'='*74}
PIPELINE COMPLETE
{'='*74}

  data          1,230 raw → 1,226 clean → {mm['n_features']} model features
  labels        Track A on all 1,226 (baseline) | Track B on 200 (target)
  judgments     900 blind pairwise, {fit['self_consistency']:.1%} self-consistent
  validation    Spearman(A,B) = {fit['spearman_A_B']:.3f}; {1-fit['linear_r2']:.0%} of the target is
                not a linear image of the rubric
  model         {mm['final_model']}
                CV  ρ = {ev.loc[mm['final_model'],'cv_spearman_mean']:.3f} ± {ev.loc[mm['final_model'],'cv_spearman_sd']:.3f}
                LORO ρ = {ev.loc[mm['final_model'],'loro_spearman_mean']:.3f} ± {ev.loc[mm['final_model'],'loro_spearman_sd']:.3f}
                rubric baseline, same folds: {ev.loc['Track A composite','loro_spearman_mean']:.3f}
  rankings      global 1,226 | regional 6 | country {rm['n_eligible_countries']}
  headline      accessibility {cmp.loc['B11_accessibility','ratio']:.1f}× its declared weight;
                technical performance {cmp.loc['B9_technical_perf','ratio']:.2f}×
  artefacts     {len(inv)} files, {inv.kb.sum()/1024:.1f} MB
  verification  {(V.status=='PASS').sum()}/{len(V)} plan conditions PASS

{'='*74}
""")


PIPELINE COMPLETE

  data          1,230 raw → 1,226 clean → 78 model features
  labels        Track A on all 1,226 (baseline) | Track B on 200 (target)
  judgments     900 blind pairwise, 96.7% self-consistent
  validation    Spearman(A,B) = 0.814; 40% of the target is
                not a linear image of the rubric
  model         LightGBM+labelwt
                CV  ρ = 0.873 ± 0.022
                LORO ρ = 0.844 ± 0.021
                rubric baseline, same folds: 0.774
  rankings      global 1,226 | regional 6 | country 19
  headline      accessibility 0.2× its declared weight;
                technical performance 0.86×
  artefacts     148 files, 10.8 MB
  verification  8/8 plan conditions PASS


